# v29 学習ノート — 系列モデル(per-well TCN+Attn) on v28特徴 → v28 GBM と残差ブレンド
- 入力: `v28/artifacts_v28/data/train.parquet`(v28 create の特徴をそのまま流用。**createは不要**)。
- モデル: 非因果 dilated TCN(受容野±510行)+自己注意1層。ヘッド= **mse** と **mdn2**(2成分混合密度→posterior mean=双峰datumのhedge)。
- fold は **v28 GBM と同一**(行順そのまま GroupKFold(5) by well)→ OOF同士の正直なブレンド重み探索。
- 評価は v28 と同じ **OOF overall(apply_pp)**。基準= v28 GBM ≈7.0(真のCV)。
- 工夫: 絶対TVT列の自動lk残差化 / robust scale / channel-dropout+noise+random-crop / EMA / cosine+warmup / bf16 / seed×headアンサンブル。
- 再開可: `artifacts_v29/nn/*.pt` があれば学習skip。GBM OOF は `gbm_oof.npy` にキャッシュ。


In [ ]:
# ===== 設定 =====
import os, json, math, time, gc, copy, glob, warnings
import joblib
import numpy as np, pandas as pd
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore")

V28_ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v28\artifacts_v28")
PARQUET = V28_ART / "data" / "train.parquet"
ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v29\artifacts_v29")
(ART / "nn").mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
TOP_K    = 192      # GBM gain importance 上位K列を NN 入力に(FORCE_FEATS優先)
FORCE_FEATS = ["md_since", "pf_ancc", "pf_ancc_2", "pf_ancc_3",
               "pf_self", "pf_self_2", "pf_self_3", "pf_nbr", "pf_nbr_2", "pf_nbr_3",
               "pf_self_vs_tw", "pf_self_vs_tw_2", "pf_self_vs_tw_3",
               "pf_nbr_vs_tw", "pf_nbr_vs_tw_2", "pf_nbr_vs_tw_3",
               "pf_self_mean", "pf_nbr_mean", "pf_self_pstd", "pf_nbr_pstd",
               "pf_self_mean_vs_tw", "pf_nbr_mean_vs_tw",
               "pf_agree_std", "pf_agree_std_2", "pf_agree_std_3",
               "pf_nbr_has", "pf_nbr_nndist", "pf_nbr_nrefs", "z_grad"]
# --- モデル ---
CH = 192; KSIZE = 5; DILATIONS = [1, 2, 4, 8, 16, 32, 64, 128]; DROP = 0.10
USE_ATTN = True; ATTN_HEADS = 4; CH_DROP = 0.10
# --- 学習 ---
HEADS = ["mse", "mdn2"]; SEEDS = [0, 1, 2]
EPOCHS = 60; LR = 3e-4; WD = 1e-4; BATCH_WELLS = 16; CROP = 1024
NOISE = 0.02; EMA_DECAY = 0.999; CLIP = 1.0; WARM_FRAC = 0.05
EVAL_TOK = 60_000   # 検証バッチの行数予算(B×Lmax上限)。VRAM逼迫時はさらに下げる
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pp_params = (json.loads((V28_ART / "pp_params.json").read_text())
             if (V28_ART / "pp_params.json").exists() else {"alpha": 1.0, "tau": 85, "w_pf": 0.09})
print("device:", DEVICE, "| torch", torch.__version__, "| pp:", pp_params)


In [ ]:
# ===== 特徴選択: v28 LightGBM の gain importance 上位 + コア強制 =====
features_all = json.loads((V28_ART / "features.json").read_text())
rank_fp = ART / "feat_rank.json"
if rank_fp.exists():
    rank = json.loads(rank_fp.read_text())
else:
    gain = np.zeros(len(features_all)); n_m = 0
    for d in sorted(glob.glob(str(V28_ART / "models" / "lightgbm-*"))):
        for fp in sorted(glob.glob(str(Path(d) / "fold*.pkl"))):
            m = joblib.load(fp)
            gain += m.booster_.feature_importance(importance_type="gain"); n_m += 1
    rank = [features_all[i] for i in np.argsort(-gain)]
    rank_fp.write_text(json.dumps(rank)); print("importance from", n_m, "lightgbm fold models")
rank = [c for c in rank if c != "last_known_tvt"]   # lk自体は残差化で0になるだけなので除外
feats_nn = [c for c in FORCE_FEATS if c in features_all]
for c in rank:
    if len(feats_nn) >= TOP_K: break
    if c not in feats_nn: feats_nn.append(c)
print("NN特徴:", len(feats_nn), "| importance top10:", rank[:10])


In [ ]:
# ===== データ読込(pyarrow 必要列のみ→事前確保numpy) + 前処理 =====
import pyarrow.parquet as pq
need = list(dict.fromkeys(["well", "target", "last_known_tvt", "md_since", "pf_ancc"] + feats_nn))
tbl = pq.read_table(PARQUET, columns=need)
N = tbl.num_rows
g      = tbl.column("well").to_pandas().to_numpy()
y      = tbl.column("target").to_numpy(zero_copy_only=False).astype(np.float64)
lk     = tbl.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds    = tbl.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
pf_abs = tbl.column("pf_ancc").to_numpy(zero_copy_only=False).astype(np.float64)
X = np.empty((N, len(feats_nn)), np.float32)
for j, c in enumerate(feats_nn):
    X[:, j] = tbl.column(c).to_numpy(zero_copy_only=False).astype(np.float32)
del tbl; gc.collect()

# fold は v28 GBM と同一(行順のまま GroupKFold(5))
folds = list(GroupKFold(n_splits=N_SPLITS).split(np.zeros(N), y, groups=g))

# 絶対TVTスケール列(corr(f,lk)>0.98)を lk 残差化 → NNが扱える小さい残差表現に
samp = np.arange(0, N, max(1, N // 120000))
abs2res = []
for j in range(X.shape[1]):
    v = X[samp, j].astype(np.float64); m = np.isfinite(v)
    ok = m.sum() > 100 and np.nanstd(v[m]) > 1e-9
    c_ = float(np.corrcoef(v[m], lk[samp][m])[0, 1]) if ok else 0.0
    a = bool(c_ > 0.98); abs2res.append(a)
    if a: X[:, j] = X[:, j] - lk.astype(np.float32)
print("abs→res 変換", int(sum(abs2res)), "列:",
      [feats_nn[j] for j in range(len(feats_nn)) if abs2res[j]][:14], "...")

med = np.nanmedian(X, axis=0)
q75 = np.nanpercentile(X, 75, axis=0); q25 = np.nanpercentile(X, 25, axis=0)
scl = np.maximum((q75 - q25) / 1.349, 1e-6)
X = (X - med) / scl
X = np.clip(np.nan_to_num(X, nan=0.0, posinf=8.0, neginf=-8.0), -8.0, 8.0).astype(np.float32)

t_mean = float(y.mean()); t_scale = float(y.std())
yn = ((y - t_mean) / t_scale).astype(np.float32)

# 井戸系列(井戸内 md_since 順, 元行indexを保持)
wid_order = pd.unique(g)
well_rows = {}
for w in wid_order:
    idx = np.where(g == w)[0]
    well_rows[w] = idx[np.argsort(mds[idx], kind="stable")]
fold_wells = [(pd.unique(g[tr]), pd.unique(g[va])) for tr, va in folds]

(ART / "norm.json").write_text(json.dumps(dict(
    features=feats_nn, abs2res=abs2res, med=med.tolist(), scale=scl.tolist(),
    t_mean=t_mean, t_scale=t_scale)))
(ART / "config.json").write_text(json.dumps(dict(
    ch=CH, ksize=KSIZE, dilations=DILATIONS, drop=DROP, use_attn=USE_ATTN,
    attn_heads=ATTN_HEADS, ch_drop=CH_DROP, n_splits=N_SPLITS)))
print("rows", N, "| wells", len(wid_order), "| K", X.shape[1],
      "| t_mean/scale", round(t_mean, 3), "/", round(t_scale, 3))


In [ ]:
# ===== モデル定義 + 学習ユーティリティ =====
class ChanLN(nn.Module):
    def __init__(self, c):
        super().__init__(); self.ln = nn.LayerNorm(c)
    def forward(self, x):  # x:(B,C,L)
        return self.ln(x.transpose(1, 2)).transpose(1, 2)

class Block(nn.Module):
    def __init__(self, c, dil, k, drop):
        super().__init__()
        self.conv = nn.Conv1d(c, c, k, padding=dil * (k - 1) // 2, dilation=dil)
        self.ln = ChanLN(c); self.pw = nn.Conv1d(c, c, 1); self.do = nn.Dropout(drop)
    def forward(self, x, m):
        h = self.conv(x * m); h = F.gelu(h); h = self.ln(h); h = self.do(self.pw(h))
        return (x + h) * m

class SelfAttn(nn.Module):
    """SDPA(flash/mem-efficient)自前実装。L×L 注意行列を実体化しない(メモリO(L))。"""
    def __init__(self, ch, heads, drop):
        super().__init__()
        self.h = heads; self.dk = ch // heads; self.p = drop
        self.qkv = nn.Linear(ch, 3 * ch); self.proj = nn.Linear(ch, ch)
    def forward(self, x, kmask):  # x:(B,L,C), kmask:(B,L) bool=有効
        B, L, C = x.shape
        q, k, v = self.qkv(x).chunk(3, -1)
        q = q.view(B, L, self.h, self.dk).transpose(1, 2)
        k = k.view(B, L, self.h, self.dk).transpose(1, 2)
        v = v.view(B, L, self.h, self.dk).transpose(1, 2)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=kmask[:, None, None, :],
                                           dropout_p=self.p if self.training else 0.0)
        return self.proj(o.transpose(1, 2).reshape(B, L, C))

class SeqNet(nn.Module):
    """非因果 dilated TCN + 自己注意(井戸全体の文脈) + head(mse=1ch / mdn2=5ch)。"""
    def __init__(self, k_in, head, ch, ksize, dilations, drop, use_attn, attn_heads, ch_drop):
        super().__init__()
        self.head = head
        self.inp = nn.Conv1d(k_in, ch, 1); self.indrop = nn.Dropout1d(ch_drop)
        self.blocks = nn.ModuleList([Block(ch, d, ksize, drop) for d in dilations])
        self.use_attn = use_attn
        if use_attn:
            self.attn = SelfAttn(ch, attn_heads, drop)
            self.aln = nn.LayerNorm(ch)
        self.out = nn.Conv1d(ch, 5 if head == "mdn2" else 1, 1)
    def forward(self, x, m):  # x:(B,K,L), m:(B,1,L)
        h = self.inp(self.indrop(x)) * m
        for b in self.blocks: h = b(h, m)
        if self.use_attn:
            q = self.aln(h.transpose(1, 2))
            a = self.attn(q, (m[:, 0, :] > 0.5))
            h = (h + a.transpose(1, 2)) * m
        return self.out(h)  # (B, 1|5, L)

def head_loss(out, yb, mb, head):
    if head == "mse":
        p = out[:, 0]
        return ((p - yb) ** 2 * mb).sum() / mb.sum(), p
    mu1, mu2 = out[:, 0], out[:, 1]
    pl = torch.sigmoid(out[:, 2])
    ls1 = out[:, 3].clamp(-3.0, 2.0); ls2 = out[:, 4].clamp(-3.0, 2.0)
    def logN(mu, ls): return -0.5 * ((yb - mu) / ls.exp()) ** 2 - ls - 0.5 * math.log(2 * math.pi)
    nll = -torch.logsumexp(torch.stack([torch.log(pl + 1e-6) + logN(mu1, ls1),
                                        torch.log(1 - pl + 1e-6) + logN(mu2, ls2)]), dim=0)
    mean = pl * mu1 + (1 - pl) * mu2
    loss = (nll * mb).sum() / mb.sum() + 0.1 * ((mean - yb) ** 2 * mb).sum() / mb.sum()
    return loss, mean

def predict_head(out, head):
    if head == "mse": return out[:, 0]
    pl = torch.sigmoid(out[:, 2]); return pl * out[:, 0] + (1 - pl) * out[:, 1]

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow: self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
    def copy_to(self, model):
        sd = model.state_dict()
        for k, v in self.shadow.items(): sd[k].copy_(v)

def make_batch(rows_list, train, rng):
    sel = []
    for r in rows_list:
        if train and len(r) > CROP:
            s = int(rng.integers(0, len(r) - CROP + 1)); r = r[s:s + CROP]
        sel.append(r)
    Lm = max(len(r) for r in sel); B = len(sel)
    xb = np.zeros((B, Lm, X.shape[1]), np.float32)
    yb = np.zeros((B, Lm), np.float32); mb = np.zeros((B, Lm), np.float32)
    for i, r in enumerate(sel):
        xb[i, :len(r)] = X[r]; yb[i, :len(r)] = yn[r]; mb[i, :len(r)] = 1.0
    return (torch.from_numpy(xb).to(DEVICE).transpose(1, 2),
            torch.from_numpy(yb).to(DEVICE), torch.from_numpy(mb).to(DEVICE), sel)

@torch.no_grad()
def predict_rows(model, head, rows_list, tok=None):
    """長さ順に並べ、B×Lmax<=tok の予算でバッチ化(パディング浪費とVRAMスパイクを抑制)。"""
    tok = tok or EVAL_TOK
    model.eval(); out = np.zeros(N, np.float64)
    order = sorted(range(len(rows_list)), key=lambda i: len(rows_list[i]))
    i = 0
    while i < len(order):
        Lm = len(rows_list[order[i]]); j = i + 1
        while j < len(order):
            Lc = max(Lm, len(rows_list[order[j]]))
            if (j - i + 1) * Lc > tok: break
            Lm = Lc; j += 1
        x, _, mb, sel = make_batch([rows_list[o] for o in order[i:j]], False, None)
        with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
            o = model(x, mb.unsqueeze(1))
        p = predict_head(o.float(), head).cpu().numpy()
        for b, r in enumerate(sel): out[r] = p[b, :len(r)]
        i = j
    return out


In [ ]:
# ===== 学習: HEADS × SEEDS × 5fold(既存 .pt はロードskip=再開可) =====
def cosine_lr(step, total, warm):
    if step < warm: return step / max(1, warm)
    t = (step - warm) / max(1, total - warm)
    return 0.5 * (1.0 + math.cos(math.pi * t))

def train_fold(head, seed, k):
    trw, vaw = fold_wells[k]
    fp = ART / "nn" / f"{head}_s{seed}_fold{k}.pt"
    model = SeqNet(X.shape[1], head, CH, KSIZE, DILATIONS, DROP, USE_ATTN, ATTN_HEADS, CH_DROP).to(DEVICE)
    va_rows = [well_rows[w] for w in vaw]
    if fp.exists():
        model.load_state_dict(torch.load(fp, map_location=DEVICE))
        return predict_rows(model, head, va_rows), "load"
    torch.manual_seed(1000 * seed + k); rng = np.random.default_rng(1000 * seed + k)
    tr_rows = [well_rows[w] for w in trw]
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    steps_ep = math.ceil(len(tr_rows) / BATCH_WELLS)
    total = steps_ep * EPOCHS; warm = int(total * WARM_FRAC)
    ema = EMA(model, EMA_DECAY); best = (1e18, None); step = 0
    va_idx = np.concatenate(va_rows)
    for ep in range(EPOCHS):
        model.train(); order = rng.permutation(len(tr_rows))
        for i in range(0, len(order), BATCH_WELLS):
            x, yb, mb, _ = make_batch([tr_rows[j] for j in order[i:i + BATCH_WELLS]], True, rng)
            if NOISE: x = x + NOISE * torch.randn_like(x)
            for pg in opt.param_groups: pg["lr"] = LR * cosine_lr(step, total, warm)
            with torch.autocast(DEVICE, dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
                out = model(x, mb.unsqueeze(1))
            loss, _ = head_loss(out.float(), yb, mb, head)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            opt.step(); ema.update(model); step += 1
        bak = copy.deepcopy(model.state_dict()); ema.copy_to(model)   # EMA重みで検証
        p = predict_rows(model, head, va_rows)
        rm = float(np.sqrt(np.mean((p[va_idx] * t_scale + t_mean - y[va_idx]) ** 2)))
        if rm < best[0]: best = (rm, copy.deepcopy(model.state_dict()))
        model.load_state_dict(bak)
    model.load_state_dict(best[1]); torch.save(best[1], fp)
    return predict_rows(model, head, va_rows), f"fit best={best[0]:.4f}"

nn_oof = {}
for head in HEADS:
    for seed in SEEDS:
        key = f"{head}_s{seed}"; oof = np.zeros(N, np.float64); t0 = time.time()
        for k in range(N_SPLITS):
            p, tag = train_fold(head, seed, k)
            va = folds[k][1]; oof[va] = p[va]
            gc.collect()
            if DEVICE == "cuda": torch.cuda.empty_cache()
            print(f"{key} fold{k} [{tag}] "
                  f"rmse={np.sqrt(np.mean((p[va] * t_scale + t_mean - y[va]) ** 2)):.4f}", flush=True)
        nn_oof[key] = oof * t_scale + t_mean
        print(f"== {key} OOF raw rmse={np.sqrt(np.mean((nn_oof[key] - y) ** 2)):.4f}"
              f" ({time.time() - t0:.0f}s)")
np.savez(ART / "nn_oof.npz", **nn_oof)


In [ ]:
# ===== NN OOF 評価(raw / apply_pp) =====
pf_res = pf_abs - lk
w_pf = float(pp_params.get("w_pf", 0.09)); tau = pp_params.get("tau", 85)
alpha = float(pp_params.get("alpha", 1.0))
def apply_pp_res(res):
    d = res * (1 - w_pf) + pf_res * w_pf
    if tau: d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(tau)))
    return d * alpha
def raw_rmse(res): return float(np.sqrt(np.mean((res - y) ** 2)))
def overall(res):  return float(np.sqrt(np.mean((apply_pp_res(res) - y) ** 2)))
for key, o in nn_oof.items():
    print(f"{key:10s} raw={raw_rmse(o):.4f}  +pp={overall(o):.4f}")
ENS = {}
for hs in ["mse", "mdn2", "all"]:
    keys = [k for k in nn_oof if hs == "all" or k.startswith(hs + "_")]
    if keys:
        ENS[hs] = (keys, np.mean([nn_oof[k] for k in keys], axis=0))
        print(f"ENS[{hs}] ({len(keys)}run) raw={raw_rmse(ENS[hs][1]):.4f}  +pp={overall(ENS[hs][1]):.4f}")


In [ ]:
# ===== v28 GBM OOF 再構成(キャッシュ: gbm_oof.npy) =====
gbm_fp = ART / "gbm_oof.npy"
if gbm_fp.exists():
    gbm_oof = np.load(gbm_fp)
else:
    feats_gbm = json.loads((V28_ART / "features.json").read_text())
    Xg = pd.read_parquet(PARQUET, columns=feats_gbm).astype("float32")
    preds = []
    for d in sorted(glob.glob(str(V28_ART / "models" / "*"))):
        if not os.path.isdir(d): continue
        po = np.zeros(N, np.float64)
        for k, (tr, va) in enumerate(folds):
            po[va] = joblib.load(Path(d) / f"fold{k}.pkl").predict(Xg.iloc[va])
        preds.append(po); print(Path(d).name, "oof done", flush=True)
    gbm_oof = np.mean(preds, axis=0); np.save(gbm_fp, gbm_oof)
    del Xg; gc.collect()
print(f"GBM OOF raw={raw_rmse(gbm_oof):.4f}  +pp={overall(gbm_oof):.4f}  (基準 ≈7.0)")


In [ ]:
# ===== ブレンド探索: (1-w)·GBM + w·NN → apply_pp(well別>10も併記) → blend.json =====
def wells_over(res, thr=10.0):
    e = apply_pp_res(res) - y
    return int(sum(1 for w in wid_order if np.sqrt(np.mean(e[well_rows[w]] ** 2)) > thr))

best = None
for hs, (keys, ens) in ENS.items():
    row = []
    for w in np.arange(0.0, 0.62, 0.02):
        sc = overall((1 - w) * gbm_oof + w * ens)
        row.append((w, sc))
        if best is None or sc < best[0]: best = (sc, hs, float(w))
    disp = "  ".join(f"{w:.1f}:{sc:.4f}" for w, sc in row if round(w * 10, 6) % 1 == 0)
    print(f"{hs:5s} {disp}")

sc, hs, w = best
keys, ens = ENS[hs]
res_b = (1 - w) * gbm_oof + w * ens
print(f"\n★best: heads={hs} w_nn={w:.2f} → OOF(apply_pp) {sc:.4f}")
print(f"   GBM単体 {overall(gbm_oof):.4f} / NN単体({hs}) {overall(ens):.4f} / Δ={sc - overall(gbm_oof):+.4f}")
print(f"   wells>10: GBM {wells_over(gbm_oof)} → blend {wells_over(res_b)}")
(ART / "blend.json").write_text(json.dumps(dict(heads=hs, keys=keys, w_nn=w, oof_pp=sc)))
print("saved:", ART / "blend.json")


In [ ]:
# ===== 追加検証: TCN(NN) × GBM の 5-fold アンサンブル比較 =====
# 前提: cell1-3(データ/fold), cell6(apply_pp_res等), cell8(wells_over) 実行済み。
# 学習セル(cell5)は再実行不要 — nn_oof.npz / gbm_oof.npy から読む。
if "nn_oof" not in globals() or not nn_oof:
    nn_oof = {k: v for k, v in np.load(ART / "nn_oof.npz").items()}
if "gbm_oof" not in globals():
    gbm_oof = np.load(ART / "gbm_oof.npy")

def fold_rmse(res):
    return [float(np.sqrt(np.mean((apply_pp_res(res)[va] - y[va]) ** 2))) for _, va in folds]

def report(name, res):
    fr = fold_rmse(res)
    print(f"{name:26s} overall={overall(res):.4f}  raw={raw_rmse(res):.4f}  wells>10={wells_over(res):3d}"
          f" | fold: " + " ".join(f"{v:.3f}" for v in fr))

nn_all = np.mean(list(nn_oof.values()), axis=0)
print("--- 単体 ---")
report("GBM(9モデル)", gbm_oof)
report(f"NN all({len(nn_oof)}run)", nn_all)
for hs in ["mse", "mdn2"]:
    keys = [k for k in nn_oof if k.startswith(hs + "_")]
    if keys:
        report(f"NN {hs}({len(keys)}run)", np.mean([nn_oof[k] for k in keys], axis=0))

print("--- 5-fold アンサンブル ---")
report("単純平均(w=0.5)", 0.5 * gbm_oof + 0.5 * nn_all)
N_GBM = 9   # GBM は 9モデル平均済みなので、等重みプールは 9:len(nn_oof) の加重に一致
pool = (gbm_oof * N_GBM + nn_all * len(nn_oof)) / (N_GBM + len(nn_oof))
report(f"等重みプール(9+{len(nn_oof)})", pool)
ws = np.arange(0.0, 1.001, 0.02)
scs = [overall((1 - w) * gbm_oof + w * nn_all) for w in ws]
wb = float(ws[int(np.argmin(scs))])
report(f"最適w={wb:.2f}", (1 - wb) * gbm_oof + wb * nn_all)
print(f"\n基準: v28 GBM train OOF CV ≈ 7.0。改善は ±0.2 seed床込みで判断。")
